# 00 - Data Quality y Análisis por Branch - Sprint 7

Este notebook responde a la observación del profesor sobre la calidad de datos y la necesidad de trabajar por branch.

## Resumen de calidad de datos

| check                      |   count | severity   | note                                                                                                        |
|:---------------------------|--------:|:-----------|:------------------------------------------------------------------------------------------------------------|
| raw_rows_total             |   11036 | info       | Filas totales en hojas TX + IP antes de limpieza                                                            |
| invalid_year_not_2024_2026 |      81 | warning    | Filas fuera del rango 2024-2026 se excluyen para consistencia temporal                                      |
| missing_kpi_label          |     217 | warning    | Filas sin etiqueta KPI válida se excluyen del modelamiento                                                  |
| missing_DATE Start         |      10 | warning    | Nulos en campo DATE Start                                                                                   |
| missing_Occur time         |      66 | warning    | Nulos en campo Occur time                                                                                   |
| missing_Priority           |      64 | warning    | Nulos en campo Priority                                                                                     |
| missing_Type of Incident   |      64 | warning    | Nulos en campo Type of Incident                                                                             |
| missing_Trouble Type       |      64 | warning    | Nulos en campo Trouble Type                                                                                 |
| missing_Incident Type      |      64 | warning    | Nulos en campo Incident Type                                                                                |
| missing_Network            |      66 | warning    | Nulos en campo Network                                                                                      |
| missing_Branch             |      65 | warning    | Nulos en campo Branch                                                                                       |
| missing_End time           |      10 | warning    | Nulos en campo End time                                                                                     |
| duration_kpi_mismatch      |    1629 | warning    | Casos donde duration_hours > sla_threshold_hours no coincide con KPI; se conserva KPI como etiqueta oficial |
| model_rows_after_filter    |   10806 | ok         | Filas válidas 2024-2026 con KPI usada en Sprint 7                                                           |
| target_over_ola_rows       |    3492 | info       | Cantidad de incidentes Over OLA                                                                             |
| target_on_time_rows        |    7314 | info       | Cantidad de incidentes On Time                                                                              |

In [ ]:
import pandas as pd
inc = pd.read_csv('../data/processed/incidents_noc_tx_ip_clean_sprint7.csv')
alarms = pd.read_csv('../data/processed/current_alarms_clean_sprint7.csv')
branch = pd.read_csv('../data/processed/branch_summary_sprint7.csv')
inc.shape, alarms.shape, branch.shape

In [ ]:
inc['label_over_ola'].value_counts(normalize=True).rename('proportion').to_frame()

## Top 10 branches por volumen

| branch     | branch_id   |   n_incidents |   over_ola |   over_ola_rate |   critical_count |   tx_count |   ip_count | top_reason   |
|:-----------|:------------|--------------:|-----------:|----------------:|-----------------:|-----------:|-----------:|:-------------|
| PIU        | branch_035  |           897 |        259 |          0.2887 |              515 |        276 |        621 | fiber_cable  |
| SAN        | branch_038  |           844 |        326 |          0.3863 |              570 |        478 |        366 | fiber_cable  |
| LIMA 1     | branch_026  |           748 |        295 |          0.3944 |              493 |        181 |        567 | fiber_cable  |
| TELEFONICA | branch_041  |           690 |        202 |          0.2928 |              583 |        508 |        182 | fiber_cable  |
| LIMA 4     | branch_029  |           589 |        219 |          0.3718 |              304 |         80 |        509 | fiber_cable  |
| HUN        | branch_019  |           552 |        200 |          0.3623 |              369 |        199 |        353 | fiber_cable  |
| LIMA 8     | branch_032  |           374 |        138 |          0.369  |              242 |        152 |        222 | fiber_cable  |
| CAJ        | branch_011  |           365 |         98 |          0.2685 |              226 |        227 |        138 | fiber_cable  |
| ARE        | branch_004  |           354 |        130 |          0.3672 |              208 |        193 |        161 | fiber_cable  |
| LIMA 2     | branch_027  |           350 |        144 |          0.4114 |              223 |         43 |        307 | fiber_cable  |

In [ ]:
branch.sort_values('n_incidents', ascending=False).head(15)[['branch','branch_id','n_incidents','over_ola','over_ola_rate','top_reason']]

In [ ]:
import matplotlib.pyplot as plt
plot_df = branch.sort_values('n_incidents', ascending=False).head(15).sort_values('over_ola_rate')
plt.figure(figsize=(8,6))
plt.barh(plot_df['branch'], plot_df['over_ola_rate'])
plt.xlabel('Tasa Over OLA')
plt.ylabel('Branch')
plt.title('Top 15 branches por volumen: tasa Over OLA')
plt.grid(axis='x', alpha=0.3)
plt.show()

## Lectura técnica

La data ya no se analiza solo globalmente. El resumen por branch permite detectar si un branch concentra más incidentes, mayor tasa Over OLA o patrones dominantes por causa.